In [10]:
import tarfile
import pickle
import glob
import io
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os
import pandas as pd
import numpy as np


#NEEDED FOR CHTC
os.environ["TORCHINDUCTOR_CACHE_DIR"] = "/tmp/torch_cache"
os.environ["USER"] = "researcher"
os.environ["LOGNAME"] = "researcher"

from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from PIL import Image

import glob
import io
import random
import os
import sys
import numpy as np
import tifffile as tiff


# Dataloader and model

In [28]:
class LoadImages(Dataset):
    def __init__(self, 
                 image_dir):
        self.image_dir = image_dir
        self.filenames = [f for f in os.listdir(image_dir) if f.endswith(('.tif', '.tiff'))] # Filter for .tif or .tiff files
    def __len__(self):
        return len(self.filenames)
    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.filenames[idx])
        img = tiff.imread(img_path).astype(np.float32)
        return torch.tensor(img).unsqueeze(0)

class ConvAutoencoder(nn.Module):
    def __init__(self):
        super(ConvAutoencoder, self).__init__()
        
        # Encoder:
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1), 
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), 
            nn.ReLU()
        )
        
        # Decoder: 
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1), 
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1), 
            nn.Sigmoid() # Use Sigmoid if input pixels are normalized [0,1]
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x




In [45]:
# device = torch.device("mps") #For mac
# device = torch.device("cuda") #if cuda
device = torch.device("cpu") #no gpu

# Load your data, this time the full sized imgages
dataset = LoadImages(image_dir="./tutorial_1_inputs/")
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Load your trained model
model = ConvAutoencoder().to(device)
checkpoint = torch.load('./tutorial_1_outputs/tutorial_1_inputs_model.pth') #load model from saved checkpoint
model.load_state_dict(checkpoint['model_state_dict'])

# Inferencing
model.eval()
model.to(device)

torch.manual_seed(777)

# Load images
dataset = LoadImages(image_dir="./tutorial_1_inputs/")
loader = DataLoader(dataset, batch_size=1)


with torch.no_grad(): # Disable gradient calculation (saves memory)
    for images in loader:
        images = images.view(-1, 1, 2048, 2048)
        images = images.to(device)
        predictions = model(images) 
        
        probs = torch.sigmoid(predictions).squeeze() # Get values 0.0 to 1.0
        pred_mask = (probs > 0.7).cpu().numpy().astype(np.uint8) * 255
        pred_mask_img = Image.fromarray(pred_mask)

FileNotFoundError: [Errno 2] No such file or directory: './tutorial_1_outputs/tutorial_1_inputs_model.pth'

In [ ]:
# Show the last image created
pred_mask_img.show()
